# Lab 3.3 &mdash; Build a Real StateGraph

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 45 min &nbsp;|&nbsp; **Day 1 &middot; Module 3 &mdash; Memory, State &amp; the LangGraph Substrate**

### What you'll do
- Declare state as a <code>TypedDict</code> with <code>Annotated</code> reducers
- Write nodes that return <i>partial</i> state, and let the reducers merge it
- Wire edges, a conditional edge and a cycle, then <code>compile()</code>
- Watch it run with <code>stream()</code>, one node at a time

> **How this lab works.** You write real LangChain and LangGraph code. Fill every `BLANK`,
> then run the **Self-check** cell under each section &mdash; those check the *objects you built*
> (a bound tool, a compiled graph, an emitted tool call), so they are deterministic and do not
> depend on the model. Cells marked **Run it for real** put your code in front of the sandbox
> model; that is the part worth watching. The score line is feedback, not a grade.

> **Builds on Lab 3.1.** A checkpointer gave you memory. A graph gives you a state you
> can declare, inspect and reason about &mdash; which is what Modules 5 and 9 build on.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-3-03")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError as exc:
        print(f"(a blank above is still unfilled: {exc} -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nSelf-check: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

# The served model reasons before it answers, and the reasoning is billed as completion
# tokens: 24.1s / 980 tokens with it on, 0.7s / 29 with it off, for the same answer. Off is
# the default here because you will make a lot of calls today. Pass think=True to see the
# difference for yourself -- and note that prompts written as an explicit ordered procedure
# survive thinking being off, while vague ones do not.
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm_cache = {}
def get_llm(temperature: float = 0.0, think: bool = False):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    from langchain_openai import ChatOpenAI
    key = (temperature, think)
    if key not in _llm_cache:
        kwargs = {} if think else {"extra_body": NO_THINK}
        _llm_cache[key] = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                                     api_key=LLM_API_KEY, temperature=temperature, **kwargs)
    return _llm_cache[key]

def ask(prompt: str, system: str | None = None, think: bool = False) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm(think=think).invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

def show_messages(messages, width: int = 88) -> None:
    """Print a message list the way a trace reads: type, content, and any tool calls."""
    for m in messages:
        kind = getattr(m, "type", "?")
        body = str(getattr(m, "content", "")).replace("\n", " ")[:width]
        calls = getattr(m, "tool_calls", None)
        line = f"  [{kind:9}] {body}"
        if calls:
            line += "  -> calls: " + ", ".join(f"{c['name']}({c['args']})" for c in calls)
        print(line)

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- the object-level self-checks still work)")

In [ ]:
# ------------------------------------------------- the case file (synthetic, self-contained)
# One domain runs through all five Module 3 labs: payment exceptions on a small ledger.
# Nothing here is real data and nothing leaves this notebook.

LEDGER = {
    "PMT-1001": {"amount": 250000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "settled",  "value_date": "2026-09-01", "reason_code": None},
    "PMT-1002": {"amount":  48250.75, "ccy": "EUR", "counterparty": "ACME-EU",
                 "status": "failed",   "value_date": "2026-09-02", "reason_code": "INSUFFICIENT_FUNDS"},
    "PMT-1003": {"amount": 990000.00, "ccy": "USD", "counterparty": "ZENITH",
                 "status": "held",     "value_date": "2026-09-02", "reason_code": "LIMIT_BREACH"},
    "PMT-1004": {"amount":   1200.00, "ccy": "GBP", "counterparty": "ACME-UK",
                 "status": "failed",   "value_date": "2026-09-03", "reason_code": "INVALID_IBAN"},
    "PMT-1005": {"amount": 750000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "held",     "value_date": "2026-09-03", "reason_code": "SANCTIONS_REVIEW"},
}

POLICY = {
    "INSUFFICIENT_FUNDS": "Retry once after 24h. If it fails again, notify the client desk. No manual funding.",
    "LIMIT_BREACH":       "Payments above USD 500,000 need Treasury approval before release.",
    "INVALID_IBAN":       "Return to originator with code R04. Never repair beneficiary details in-house.",
    "SANCTIONS_REVIEW":   "Hold. Compliance decides. Operations must not release or cancel.",
}

# Which reason codes may an agent resolve on its own, and which need a human?
NEEDS_HUMAN = {"LIMIT_BREACH", "SANCTIONS_REVIEW"}

print(f"{len(LEDGER)} payments, {len(POLICY)} policy rules loaded")

In [ ]:
# ------------------------------------------------- the case tools, carried through 3.3 - 3.5
def read_ledger_record(ref: str) -> dict:
    rec = LEDGER.get(ref)
    return {"ref": ref, **rec} if rec else {"ref": ref, "error": "not_found"}

def read_policy_text(reason_code: str | None) -> str:
    return POLICY.get(reason_code, "no policy applies")

print("case helpers loaded")

## Concept

`create_agent` is one fixed loop: model, tools, repeat. When the control flow is yours &mdash; branch
here, loop there, stop for a human &mdash; you need the graph underneath it.

A LangGraph `StateGraph` has three parts and no more:

| Part | What it is |
|---|---|
| **state** | a `TypedDict`; each field may carry a **reducer** saying how updates merge |
| **nodes** | plain functions `state -> partial state` |
| **edges** | fixed (`add_edge`) or chosen at runtime (`add_conditional_edges`) |

Then `compile()` gives you a runnable with the same `.invoke()` / `.stream()` interface as
everything else you have built today.

The part people get wrong is the reducer, so start there.

## Section 1 &mdash; State, and how updates merge

By default a node's return value **overwrites** the field. That is right for `answer` and wrong
for `findings` &mdash; you want those to accumulate. `Annotated[list, add]` says "append, do not
replace".

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from operator import add
from langgraph.graph import StateGraph, START, END

def accumulate(old: list, new: list) -> list:
    """The reducer for `findings`: how one node's update merges with what is already there."""
    return BLANK                                # TODO: combine them so BOTH nodes' findings survive


class CaseState(TypedDict):
    ref: str                                    # set once, overwritten if a node returns it
    findings: Annotated[list, accumulate]       # merged by your reducer, not replaced
    steps: int
    needs_human: bool
    answer: str | None

In [ ]:
# --- Self-check: Section 1   (the reducer, exercised through a real one-node graph -- no model)
def _reducer_of(field):
    """The reducer LangGraph will use for one field of CaseState."""
    from typing import get_type_hints
    hints = get_type_hints(CaseState, include_extras=True)
    meta = getattr(hints[field], "__metadata__", ())
    return meta[0] if meta else None

def _two_appends():
    """Two nodes, each returning one finding. Does the state end up with both?"""
    g = StateGraph(CaseState)
    g.add_node("a", lambda s: {"findings": ["from a"], "steps": 1})
    g.add_node("b", lambda s: {"findings": ["from b"], "steps": 2})
    g.add_edge(START, "a"); g.add_edge("a", "b"); g.add_edge("b", END)
    return g.compile().invoke({"ref": "PMT-1005", "findings": [], "steps": 0,
                               "needs_human": False, "answer": None})

check("findings declares a reducer",
      lambda: _reducer_of("findings") is not None,
      "without one, the second node's findings replace the first node's")
check("the reducer merges two updates instead of replacing",
      lambda: accumulate(["a"], ["b"]) == ["a", "b"],
      "returning just `new` is the bug this whole section exists to prevent")
check("two nodes' findings both survive",
      lambda: _two_appends()["findings"] == ["from a", "from b"],
      "this is what the reducer is FOR -- run it and see")
check("a field with no reducer is overwritten",
      lambda: _two_appends()["steps"] == 2,
      "steps has no reducer, so the last write wins -- that is the default, and it is fine here")
check("the untouched fields are still there",
      lambda: _two_appends()["ref"] == "PMT-1005")

## Section 2 &mdash; Nodes return partial state

A node receives the whole state and returns **only the keys it changed**. LangGraph merges the
rest for you, using the reducers from Section 1. Returning the whole state is the other common
beginner mistake: it works, until two nodes run and one silently undoes the other.

In [ ]:
def read_ledger(state: CaseState) -> dict:
    """Look the payment up and record what we found."""
    rec = read_ledger_record(state["ref"])
    return {"findings": [f"ledger: status={rec.get('status')} rc={rec.get('reason_code')}"],
            "steps": state["steps"] + 1,
            "needs_human": rec.get("reason_code") in NEEDS_HUMAN}


def read_policy(state: CaseState) -> dict:
    """Look up the policy for whatever reason code the ledger gave us."""
    rec = read_ledger_record(state["ref"])
    return {"findings": [f"policy: {read_policy_text(rec.get('reason_code'))}"],
            "steps": state["steps"] + 1}


def write_note(state: CaseState) -> dict:
    """Compose the answer from what is in state -- and nothing else."""
    who = "a human must decide" if state["needs_human"] else "operations may act"
    return {"answer": BLANK}          # TODO: an answer built from the findings and `who`

In [ ]:
# --- Self-check: Section 2   (nodes are plain functions -- call them directly, no model)
_s = {"ref": "PMT-1005", "findings": [], "steps": 0, "needs_human": False, "answer": None}

check("a node returns only what it changed",
      lambda: set(read_ledger(_s)) == {"findings", "steps", "needs_human"},
      "returning the whole state is how one node silently undoes another")
check("the ledger node finds the reason code",
      lambda: "SANCTIONS_REVIEW" in read_ledger(_s)["findings"][0])
check("it sets needs_human for a sanctions hold",
      lambda: read_ledger(_s)["needs_human"] is True)
check("...and not for an ordinary failure",
      lambda: read_ledger({**_s, "ref": "PMT-1002"})["needs_human"] is False)
check("the policy node returns the policy text",
      lambda: "Compliance" in read_policy(_s)["findings"][0])
check("write_note uses the findings it was given",
      lambda: "ledger:" in write_note({**_s, "findings": ["ledger: x"], "needs_human": True})["answer"])
check("write_note says who decides",
      lambda: "human" in write_note({**_s, "findings": ["x"], "needs_human": True})["answer"])

## Section 3 &mdash; Edges, a condition, and a cycle

`add_edge(a, b)` always goes to `b`. `add_conditional_edges(a, fn, mapping)` calls `fn(state)` and
goes wherever it says. A cycle is just an edge that points backwards &mdash; which is why the step
budget is not optional.

In [ ]:
MAX_STEPS = 6

def enough(state: CaseState) -> str:
    """Have we gathered enough to answer? Returns the KEY of the next branch."""
    if state["steps"] >= MAX_STEPS:
        return "write_note"
    return "write_note" if len(state["findings"]) >= 2 else "read_ledger"


def build_graph():
    g = StateGraph(CaseState)
    g.add_node("read_ledger", read_ledger)
    g.add_node("read_policy", read_policy)
    g.add_node("write_note", write_note)

    g.add_edge(START, "read_ledger")
    g.add_edge("read_ledger", "read_policy")
    g.add_conditional_edges("read_policy", enough,
                            {"write_note": "write_note",
                             "read_ledger": BLANK})     # TODO: where does "not enough yet" go?
    g.add_edge("write_note", END)
    return g.compile()


def fresh(ref: str) -> dict:
    return {"ref": ref, "findings": [], "steps": 0, "needs_human": False, "answer": None}

In [ ]:
# --- Self-check: Section 3   (a REAL compiled graph, running -- still no model)
def _run(ref="PMT-1005"):
    return build_graph().invoke(fresh(ref))

check("the graph compiles",              lambda: build_graph() is not None)
check("it runs to an answer",            lambda: _run()["answer"] is not None)
check("both nodes contributed findings", lambda: len(_run()["findings"]) >= 2)
check("the reducer accumulated them",    lambda: any("ledger:" in f for f in _run()["findings"])
                                                 and any("policy:" in f for f in _run()["findings"]))
check("the sanctions case needs a human",
      lambda: _run("PMT-1005")["needs_human"] is True)
check("an ordinary failure does not",
      lambda: _run("PMT-1002")["needs_human"] is False)
check("the cycle terminates",            lambda: _run()["steps"] <= MAX_STEPS,
      "a backward edge with no budget is an infinite loop, and the graph will happily run it")
check("the conditional edge can actually loop",
      lambda: enough({"steps": 0, "findings": []}) == "read_ledger",
      "if both branches go forward you have written a straight line, not a cycle")

## Watch it run

`stream()` yields one entry per node as it completes, which is the cheapest debugger you will
ever have for a graph.

In [ ]:
def _trace():
    for chunk in build_graph().stream(fresh("PMT-1005")):
        for node, update in chunk.items():
            print(f"  {node:14} -> {list(update)}")
    print("\nfinal answer:")
    print("  " + str(build_graph().invoke(fresh("PMT-1005"))["answer"])[:300])
guard(_trace)

## Run it for real &mdash; put the model in a node

Nothing so far needed a model, which is the point: **the graph is deterministic scaffolding, and
you can test all of it offline.** Now add one node that does need one. Note that it reads only
from state, and returns only a partial update &mdash; exactly like the others.

In [ ]:
if llm_ready():
    def _with_model():
        def draft_note(state: CaseState) -> dict:
            who = "a human must decide" if state["needs_human"] else "operations may act"
            text = ask("Write one line telling the operations desk what happens next. "
                       "Use only these facts; do not add any.\n"
                       f"CASE: {state['ref']}\nAUTHORITY: {who}\n"
                       f"FINDINGS: {state['findings']}")
            return {"answer": text.strip()}

        g = StateGraph(CaseState)
        g.add_node("read_ledger", read_ledger)
        g.add_node("read_policy", read_policy)
        g.add_node("draft_note", draft_note)
        g.add_edge(START, "read_ledger")
        g.add_edge("read_ledger", "read_policy")
        g.add_conditional_edges("read_policy", lambda s: "draft_note" if len(s["findings"]) >= 2
                                else "read_ledger",
                                {"draft_note": "draft_note", "read_ledger": "read_ledger"})
        g.add_edge("draft_note", END)

        app = g.compile()
        for ref in ("PMT-1005", "PMT-1002"):
            out = app.invoke(fresh(ref))
            print(f"{ref}: needs_human={out['needs_human']}")
            print(f"          {out['answer'][:200]}\n")
    guard(_with_model)

### Read it

Three things worth taking away.

1. **The graph is testable without a model.** Every self-check in this lab ran a real compiled
   `StateGraph` and asserted on real merged state, offline and deterministically. That is not a
   trick of the lab &mdash; it is how you should test agent control flow generally. Put the model in
   one node, and everything around it stays ordinary software.
2. **The reducer is the design.** `findings` accumulates because you said so; `steps` overwrites
   because you did not. Get that wrong and the bug looks like "the second agent lost the first
   agent's work", which is Lab 3.5's subject.
3. **The cycle needs the budget.** `enough` checks `MAX_STEPS` before it checks anything else. A
   backward edge with no ceiling is an infinite loop that LangGraph will run for you, cheerfully,
   until something else stops it.

In [ ]:
score()

## Your turn

1. Add a `needs_escalation` node that runs only when `needs_human` is true, and route to it with
   a second conditional edge. Confirm PMT-1002 never enters it.
2. Give `steps` the reducer `add` instead of leaving it to overwrite, and change the nodes to
   return `{"steps": 1}`. Which do you prefer, and what happens if two nodes ever run in
   parallel?
3. `read_policy` calls `read_ledger_record` again, because the reason code was never put in state.
   Add a `reason_code` field, set it in `read_ledger`, and read it in `read_policy`. That is the
   difference between passing state and re-fetching it &mdash; and it is exactly what Module 5's
   multi-agent graphs depend on.